In [5]:
##### Creation images modifiées ########

import os
import cv2
import numpy as np
import random

# Applique une légère rotation aléatoire à une image, puis recadre l'image pour éviter les bordures vides.
def rotate_and_crop(image, angle_range=(-4,4)):    
    # Génère un angle aléatoire dans la plage spécifiée (en degrés)
    angle = random.uniform(*angle_range)
    h, w = image.shape[:2]
    center = (w // 2, h // 2)

    # Calcule la matrice de transformation pour la rotation autour du centre
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1)

    # Applique la rotation à l'image, en utilisant la réplication des bords pour remplir les zones vides
    rotated_image = cv2.warpAffine(image, rotation_matrix, (w, h), borderMode=cv2.BORDER_REPLICATE)

    # Convertit l'angle en radians pour les calculs trigonométriques
    radians = np.deg2rad(abs(angle))
    cos_a = np.cos(radians)
    sin_a = np.sin(radians)

    # Calcule les dimensions du plus grand rectangle inscriptible dans l'image après rotation
    new_w = int(w * cos_a - h * sin_a)
    new_h = int(h * cos_a - w * sin_a)

    # Vérifie que les nouvelles dimensions ne dépassent pas celles de l'image d'origine
    new_w = min(new_w, w)
    new_h = min(new_h, h)

    # Calcule les coordonnées du coin supérieur gauche et inférieur droit pour centrer le recadrage
    crop_x1 = (w - new_w) // 2
    crop_y1 = (h - new_h) // 2
    crop_x2 = crop_x1 + new_w
    crop_y2 = crop_y1 + new_h

    # Recadre l'image pour supprimer les bords artificiels ajoutés par la rotation
    cropped_image = rotated_image[crop_y1:crop_y2, crop_x1:crop_x2]

    return cropped_image

# Effectue un zoom avant aléatoire sur l'image.
def zoom_in(image, zoom_range=(1.0, 1.3)):
    scale = random.uniform(*zoom_range)
    h, w = image.shape[:2]
    new_h, new_w = int(h / scale), int(w / scale)

    # Calcul de la zone centrale à découper
    start_x, start_y = (w - new_w) // 2, (h - new_h) // 2
    cropped = image[start_y:start_y + new_h, start_x:start_x + new_w]
    
    # Redimensionnement à la taille d'origine
    return cv2.resize(cropped, (w, h), interpolation=cv2.INTER_CUBIC)

# Ajuste la luminosité et le contraste de l'image.
def adjust_brightness_contrast(image, alpha_range=(0.9, 1.1), beta_range=(-20, 20)):
    alpha = random.uniform(*alpha_range)
    beta = random.randint(*beta_range)
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

# Modifie légèrement les couleurs via l’espace HSV.
def modify_colors(image, hue_shift=(-10, 10), saturation_scale=(0.9, 1.1)):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + random.randint(*hue_shift)) % 180  # Teinte
    hsv[:, :, 1] *= random.uniform(*saturation_scale)                 # Saturation
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

# Ajoute un effet de lumière douce simulant une source lumineuse.
def add_light_effect(image, intensity_range=(0.2, 0.4)):
    h, w = image.shape[:2]
    mask = np.ones((h, w), dtype=np.float32)

    # Position aléatoire de la lumière
    x, y = random.randint(w//4, 3*w//4), random.randint(h//4, 3*h//4)
    radius = random.randint(h//3, h//2)

    # Création du dégradé circulaire
    for i in range(h):
        for j in range(w):
            dist = np.sqrt((i - y) ** 2 + (j - x) ** 2)
            mask[i, j] = min(1.0, 1.0 + random.uniform(*intensity_range) * (radius - dist) / radius)

    # Lissage du dégradé
    mask = cv2.GaussianBlur(mask, (101, 101), 0)
    mask = np.repeat(mask[:, :, np.newaxis], 3, axis=2)

    # Application de l’effet de lumière
    brightened_image = cv2.addWeighted(image.astype(np.float32), 1.0, mask * 255, 0.2, 0)
    return np.clip(brightened_image, 0, 255).astype(np.uint8)

# Ajouter une ombre douce à l'image avec un dégradé.
def add_shadow_effect(image, intensity_range=(-0.3, -0.1)):
    h, w = image.shape[:2]
    mask = np.ones((h, w), dtype=np.float32)

    # Position aléatoire de l’ombre
    x, y = random.randint(w//4, 3*w//4), random.randint(h//4, 3*h//4)
    radius = random.randint(h//3, h//2)

    # Création du dégradé pour simuler l’ombre
    for i in range(h):
        for j in range(w):
            dist = np.sqrt((i - y) ** 2 + (j - x) ** 2)
            mask[i, j] = max(0.7, 1.0 + random.uniform(*intensity_range) * (radius - dist) / radius)

    mask = cv2.GaussianBlur(mask, (101, 101), 0)
    mask = np.repeat(mask[:, :, np.newaxis], 3, axis=2)

    # Application de l’ombre à l’image
    shadowed_image = (image * mask).astype(np.uint8)
    return shadowed_image

# Applique un effet de déformation perspective (skew) 
def skew_image(image, max_skew=0.05):
    h, w = image.shape[:2]

    # Définition des points sources et destinations modifiés aléatoirement
    src_pts = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
    dst_pts = src_pts + np.float32([
        [random.uniform(-max_skew*w, max_skew*w), random.uniform(-max_skew*h, max_skew*h)],
        [random.uniform(-max_skew*w, max_skew*w), random.uniform(-max_skew*h, max_skew*h)],
        [random.uniform(-max_skew*w, max_skew*w), random.uniform(-max_skew*h, max_skew*h)],
        [random.uniform(-max_skew*w, max_skew*w), random.uniform(-max_skew*h, max_skew*h)]
    ])

    matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
    return cv2.warpPerspective(image, matrix, (w, h), borderMode=cv2.BORDER_REFLECT)

# Applique entre 1 et 4 transformations aléatoires à une image.
def apply_random_transformations(image):
    transformations = [
        rotate_and_crop, zoom_in, adjust_brightness_contrast, modify_colors,
        add_light_effect, add_shadow_effect, skew_image
    ]
    
    num_modifications = random.randint(1, 4) 
    random.shuffle(transformations)  # Mélange des transformations
    
    for i in range(num_modifications):
        image = transformations[i](image)  # Application successive
    
    return image

# Génère des variations transformées d’images dans un répertoire donné.
def generate_variations(input_dir, output_dir, num_variations):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(input_dir):
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(input_dir, filename)
            image = cv2.imread(img_path)

            if image is None:
                print(f"Erreur de lecture : {filename}")
                continue

            base_name, ext = os.path.splitext(filename)

            # Liste des transformations disponibles
            transformations = [rotate_and_crop, zoom_in, adjust_brightness_contrast, modify_colors, add_light_effect, add_shadow_effect, skew_image]
            for j in range(6) :  # Répéter les séries de transformations
                for i in range(7) :   # Une transformation à la fois
                    modified_image =  transformations[i](image)
                    output_path = os.path.join(output_dir, f"{j}{base_name}_{transformations[i].__name__}{ext}")
                    cv2.imwrite(output_path, modified_image)
                    print(transformations[i])

                # Appliquer des combinaisons aléatoires
                for i in range(num_variations):
                    modified_image = apply_random_transformations(image)
                    output_path = os.path.join(output_dir, f"{j}{base_name}_mod_{i}{ext}")
                    cv2.imwrite(output_path, modified_image)
                    print("ok2")

            print(f"Variations générées pour {filename}")

    print("Toutes les modifications sont terminées !")



# Lancement du traitement sur un jeu de données
generate_variations(r"C:\Insa\cours4a\projetmulti\test_images\donnees2\originales\classe2", r"C:\Insa\cours4a\projetmulti\test_images\donnees2\train\classe2", num_variations=2)


<function rotate_and_crop at 0x0000027B23C6EDD0>
<function zoom_in at 0x0000027B23C6EF80>
<function adjust_brightness_contrast at 0x0000027B0C848550>
<function modify_colors at 0x0000027B0B03F5B0>
<function add_light_effect at 0x0000027B0C848AF0>
<function add_shadow_effect at 0x0000027B23B5BC70>
<function skew_image at 0x0000027B23B5BD00>
Variations générées pour Extime643.9ms-gain3.2x-5X- red267_4807.jpg
Toutes les modifications sont terminées !
